In [5]:
from langchain_community.document_loaders import PyPDFLoader
import os

# Use relative path from the notebook location (testing folder)
file_path = "../data/amtg_handbook.pdf"

print(f"Loading PDF from: {file_path}")
print(f"File exists: {os.path.exists(file_path)}")

loader = PyPDFLoader(file_path)

Loading PDF from: ../data/amtg_handbook.pdf
File exists: True


In [6]:
# Load the PDF documents
documents = loader.load()

print(f"Loaded {len(documents)} pages from the PDF")
print(f"\nFirst page preview (first 500 characters):")
print(documents[0].page_content[:500] if documents else "No content loaded")

Loaded 677 pages from the PDF

First page preview (first 500 characters):



In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(documents)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 3104 sub-documents.


In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [12]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

In [13]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [16]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['e8a3273a-013c-417a-8b31-533117a2897c', '2e94203e-20d1-415b-916a-13108a1e47d1', '99b52a22-3951-4a7b-a14a-9e2310aeb1f7']


In [17]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [18]:
from langchain.agents import create_agent

tools = [retrieve_context]
# If desired, specify custom instructions
prompt = (
    "You have access to a tool that retrieves context from a aviation document. "
    "Use the tool to help answer user queries."
)
agent = create_agent(model, tools, system_prompt=prompt)

In [19]:
query = (
    "What is Aircraft Maintenance Technician General Handbook about?"
)

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================
What is Aircraft Maintenance Technician General Handbook about?
================================== Ai Message ==================================
thought
The user is asking a question about the content of the document. I should use the `retrieve_context` tool to find relevant information in the document and then answer the question based on the retrieved context.The user is asking what the "Aircraft Maintenance Technician General Handbook" is about. I need to use the `retrieve_context` tool to find the relevant information within the handbook to answer this question. I will pass the user's query to the tool.I need to answer the user's question about the "Aircraft Maintenance Technician General Handbook". The best way to do this is to use the `retrieve_context` tool to search the document for relevant information. I will use the user's query as the input for the tool.
================================ Tool Ca